# Notebook 04 — Modeling
**EPPS–American Airlines Data Challenge — GROW 26.2**

Trains Logistic Regression (baseline) + XGBoost (primary model)
on the sequence-level dataset with `is_risky` as the target.

**Outputs:**
- `../outputs/models/baseline_lr.pkl`
- `../outputs/models/xgboost_final.json`
- `../data/processed/sequences_scored.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings, pickle
warnings.filterwarnings('ignore')

from sklearn.linear_model       import LogisticRegression
from sklearn.ensemble           import RandomForestClassifier
from sklearn.preprocessing      import StandardScaler
from sklearn.pipeline           import Pipeline
from sklearn.model_selection    import StratifiedKFold, cross_validate
from sklearn.metrics            import (classification_report, confusion_matrix,
                                        average_precision_score, roc_auc_score,
                                        precision_recall_curve, roc_curve)
from imblearn.over_sampling     import SMOTE
from xgboost                    import XGBClassifier

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

PROC_PATH    = '../data/processed/'
MODELS_PATH  = '../outputs/models/'
FIGURES_PATH = '../outputs/figures/'
RESULTS_PATH = '../outputs/results/'
os.makedirs(MODELS_PATH,  exist_ok=True)
os.makedirs(FIGURES_PATH, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)

print('✅ Imports OK')

## 1 — Load Sequences

In [ ]:
df = pd.read_csv(f'{PROC_PATH}sequences.csv', parse_dates=['Date'])
print(f'Sequences loaded : {df.shape}')
print(f'is_risky dist    : {df["is_risky"].value_counts().to_dict()}')

## 2 — Prepare Features & Target

In [ ]:
# Columns to exclude from features
ID_COLS = ['airport_A','airport_B','Date','delayed_A','delayed_B','is_risky']
OBJ_COLS = [c for c in df.columns if df[c].dtype == 'object']

EXCLUDE = ID_COLS + OBJ_COLS
FEAT_COLS = [c for c in df.columns if c not in EXCLUDE]

print(f'Feature columns : {len(FEAT_COLS)}')
print(f'Object cols excluded: {OBJ_COLS}')
print()

X = df[FEAT_COLS].fillna(0).astype(float)
y = df['is_risky'].astype(int)

print(f'X shape : {X.shape}')
print(f'y dist  : {y.value_counts().to_dict()}')

neg, pos = (y==0).sum(), (y==1).sum()
SCALE_POS_WEIGHT = neg / pos
print(f'scale_pos_weight : {SCALE_POS_WEIGHT:.2f}')

## 3 — Train/Test Split (Time-Based)

We split chronologically — earlier months for training, later months for testing.
This prevents data leakage from future flights informing past predictions.

In [ ]:
df_sorted = df.copy()
df_sorted = df_sorted.sort_values('Date').reset_index(drop=True)

# 70/30 chronological split
split_idx = int(len(df_sorted) * 0.70)
train_idx = df_sorted.index[:split_idx]
test_idx  = df_sorted.index[split_idx:]

X_train = X.loc[train_idx].reset_index(drop=True)
X_test  = X.loc[test_idx].reset_index(drop=True)
y_train = y.loc[train_idx].reset_index(drop=True)
y_test  = y.loc[test_idx].reset_index(drop=True)

print(f'Train size : {len(X_train):,} | Risky: {y_train.sum()} ({y_train.mean()*100:.1f}%)')
print(f'Test  size : {len(X_test):,}  | Risky: {y_test.sum()}  ({y_test.mean()*100:.1f}%)')
print(f'Train dates: {df_sorted.loc[train_idx,"Date"].min().date()} → {df_sorted.loc[train_idx,"Date"].max().date()}')
print(f'Test  dates: {df_sorted.loc[test_idx, "Date"].min().date()} → {df_sorted.loc[test_idx, "Date"].max().date()}')

## 4 — Apply SMOTE on Training Set

In [ ]:
smote = SMOTE(random_state=42, k_neighbors=min(5, pos-1))
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f'Before SMOTE: {dict(pd.Series(y_train).value_counts())}')
print(f'After  SMOTE: {dict(pd.Series(y_train_sm).value_counts())}')
print(f'X_train_sm shape: {X_train_sm.shape}')

## 5 — Baseline: Logistic Regression

In [ ]:
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(max_iter=1000, random_state=42,
                                   class_weight='balanced', C=1.0))
])

lr_pipeline.fit(X_train_sm, y_train_sm)

y_pred_lr   = lr_pipeline.predict(X_test)
y_prob_lr   = lr_pipeline.predict_proba(X_test)[:, 1]

pr_auc_lr  = average_precision_score(y_test, y_prob_lr)
roc_auc_lr = roc_auc_score(y_test, y_prob_lr)

print('=== LOGISTIC REGRESSION ===')
print(f'  PR-AUC  : {pr_auc_lr:.4f}')
print(f'  ROC-AUC : {roc_auc_lr:.4f}')
print()
print(classification_report(y_test, y_pred_lr, target_names=['Not Risky','Risky']))

# Save model
with open(f'{MODELS_PATH}baseline_lr.pkl', 'wb') as f:
    pickle.dump(lr_pipeline, f)
print('💾 baseline_lr.pkl saved')

## 6 — Primary Model: XGBoost

In [ ]:
xgb_model = XGBClassifier(
    n_estimators        = 300,
    max_depth           = 4,
    learning_rate       = 0.05,
    subsample           = 0.8,
    colsample_bytree    = 0.8,
    scale_pos_weight    = SCALE_POS_WEIGHT,
    eval_metric         = 'aucpr',
    random_state        = 42,
    verbosity           = 0,
    use_label_encoder   = False
)

xgb_model.fit(
    X_train_sm, y_train_sm,
    eval_set    = [(X_test, y_test)],
    verbose     = False
)

y_pred_xgb  = xgb_model.predict(X_test)
y_prob_xgb  = xgb_model.predict_proba(X_test)[:, 1]

pr_auc_xgb  = average_precision_score(y_test, y_prob_xgb)
roc_auc_xgb = roc_auc_score(y_test, y_prob_xgb)

print('=== XGBOOST ===')
print(f'  PR-AUC  : {pr_auc_xgb:.4f}')
print(f'  ROC-AUC : {roc_auc_xgb:.4f}')
print()
print(classification_report(y_test, y_pred_xgb, target_names=['Not Risky','Risky']))

# Save model
xgb_model.save_model(f'{MODELS_PATH}xgboost_final.json')
print('💾 xgboost_final.json saved')

## 7 — PR Curve & ROC Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PR Curve
ax = axes[0]
for name, y_prob, auc in [
    ('Logistic Regression', y_prob_lr,  pr_auc_lr),
    ('XGBoost',             y_prob_xgb, pr_auc_xgb)]:
    prec, rec, _ = precision_recall_curve(y_test, y_prob)
    ax.plot(rec, prec, lw=2, label=f'{name} (PR-AUC={auc:.3f})')
baseline_pr = y_test.mean()
ax.axhline(baseline_pr, color='gray', ls='--', label=f'Baseline={baseline_pr:.3f}')
ax.set_title('Precision-Recall Curve', fontweight='bold')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.legend()
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])

# ROC Curve
ax = axes[1]
for name, y_prob, auc in [
    ('Logistic Regression', y_prob_lr,  roc_auc_lr),
    ('XGBoost',             y_prob_xgb, roc_auc_xgb)]:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    ax.plot(fpr, tpr, lw=2, label=f'{name} (AUC={auc:.3f})')
ax.plot([0,1],[0,1], 'k--', alpha=0.4)
ax.set_title('ROC Curve', fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend()
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])

plt.suptitle('Model Evaluation Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}16_pr_roc_curves.png', bbox_inches='tight')
plt.show()
print('💾 16_pr_roc_curves.png')

## 8 — Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, y_pred, title in zip(
    axes,
    [y_pred_lr, y_pred_xgb],
    ['Logistic Regression', 'XGBoost']):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Not Risky','Risky'],
                yticklabels=['Not Risky','Risky'])
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')
plt.suptitle('Confusion Matrices', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}17_confusion_matrices.png', bbox_inches='tight')
plt.show()
print('💾 17_confusion_matrices.png')

## 9 — XGBoost Feature Importance

In [ ]:
importances = pd.Series(xgb_model.feature_importances_, index=FEAT_COLS)
top20 = importances.sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 7))
colors_fi = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(top20)))
ax.barh(top20.index[::-1], top20.values[::-1], color=colors_fi[::-1], edgecolor='white')
ax.set_title('XGBoost — Top 20 Feature Importances', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}18_xgb_feature_importance.png', bbox_inches='tight')
plt.show()
print('💾 18_xgb_feature_importance.png')

## 10 — Score All Sequences & Save

In [ ]:
X_all = df[FEAT_COLS].fillna(0).astype(float)
df['risk_score_xgb'] = xgb_model.predict_proba(X_all)[:, 1]
df['risk_score_lr']  = lr_pipeline.predict_proba(X_all)[:, 1]
df['predicted_risky_xgb'] = xgb_model.predict(X_all)

df.to_csv(f'{PROC_PATH}sequences_scored.csv', index=False)
print(f'💾 sequences_scored.csv saved  ({df.shape})')
print()
print('=== MODEL SUMMARY ===')
print(f'Logistic Regression  PR-AUC={pr_auc_lr:.4f}  ROC-AUC={roc_auc_lr:.4f}')
print(f'XGBoost              PR-AUC={pr_auc_xgb:.4f}  ROC-AUC={roc_auc_xgb:.4f}')
print()
print('✅ Next → Notebook 05: Evaluation')